In [1]:
import os
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib as mpl
from argparse import Namespace
import torch
import requests
import numpy as np
import cv2
from PIL import Image
import utils_model_qwen as utils_model
import utils_gradio_qwen as utils_gradio
import utils_attn
device_map = "auto"

/home/azureuser/lvlm-interpret/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2025-04-21 14:10:27,819] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -lcufile: No such file or directory
collect2: error: ld returned 1 exit status


In [27]:
from transformers import AutoModelForImageTextToText, AutoProcessor

model_name_or_path = "Qwen/Qwen2-VL-7B-Instruct"
min_pixels = 256*28*28
max_pixels = 512*28*28

model = AutoModelForImageTextToText.from_pretrained(
    model_name_or_path,
    torch_dtype=torch.bfloat16,
    device_map=device_map
)
processor = AutoProcessor.from_pretrained(model_name_or_path, min_pixels=min_pixels, max_pixels=max_pixels)

Loading checkpoint shards: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:02<00:00,  1.93steps/s]


In [31]:
model.config.image_t

Qwen2VLConfig {
  "_attn_implementation_autoset": true,
  "architectures": [
    "Qwen2VLForConditionalGeneration"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 3584,
  "image_token_id": 151655,
  "initializer_range": 0.02,
  "intermediate_size": 18944,
  "max_position_embeddings": 32768,
  "max_window_layers": 28,
  "model_type": "qwen2_vl",
  "num_attention_heads": 28,
  "num_hidden_layers": 28,
  "num_key_value_heads": 4,
  "rms_norm_eps": 1e-06,
  "rope_scaling": {
    "mrope_section": [
      16,
      24,
      24
    ],
    "rope_type": "default",
    "type": "default"
  },
  "rope_theta": 1000000.0,
  "sliding_window": 32768,
  "tie_word_embeddings": false,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.51.3",
  "use_cache": true,
  "use_sliding_window": false,
  "video_token_id": 151656,
  "vision_config": {
    "depth": 32,
    "embed_dim": 1280,
    "hidden_act": "quick_gelu",
    "hid

In [28]:
conversation = [
    {
        "role":"user",
        "content":[
            {
                "type":"image",
                "path": "beaver.jpg"
            },
            {
                "type":"text",
                "text":"Describe this image."
            }
        ]
    }
]

inputs = processor.apply_chat_template(
    conversation,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

# Inference: Generation of the output
output_ids = model.generate(**inputs, max_new_tokens=256)
generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, output_ids)]
output_text = processor.batch_decode(generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True)
print(output_text)


["The image depicts a beaver standing upright on its hind legs. The beaver has a robust, furry body with a mix of brown and black fur. Its fur appears wet, suggesting it has been in or near water recently. The beaver's front paws are raised and positioned in front of its chest, and its back paws are planted firmly on the ground. The beaver's mouth is slightly open, revealing its teeth, which are orange in color. The background is blurred, but it appears to be an outdoor setting with some greenery and possibly water, indicating that the beaver is in its natural habitat near a body of water. The overall scene captures the beaver in a moment of alertness or curiosity."]


In [29]:
inputs.input_ids[0].shape

torch.Size([519])

In [30]:
print(inputs.input_ids[0].cpu().numpy().tolist())

[151644, 8948, 198, 2610, 525, 264, 10950, 17847, 13, 151645, 198, 151644, 872, 198, 151652, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151

In [6]:
temperature = 0.1
top_p = 0.7
max_new_tokens = 20
image = Image.open('beaver.jpg')
conversation = [
    {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": "Describe this image."}]}
]

text_prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)
inputs = processor(text=[text_prompt], images=[image], padding=True, return_tensors="pt")
inputs = {k: v.to(model.device) for k, v in inputs.items()}
outputs = model(
    **inputs,
    # temperature=temperature,
    # top_p=top_p,    
    use_cache=True,
    output_attentions=True,
)

Qwen2VLModel is using Qwen2VLSdpaAttention, but `torch.nn.functional.scaled_dot_product_attention` does not support `output_attentions=True`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


In [7]:
outputs.keys()

odict_keys(['logits', 'past_key_values', 'attentions', 'rope_deltas'])

In [5]:
temperature = 0.1
top_p = 0.7
max_new_tokens = 20

state, _ = utils_gradio.lvlm_bot(state, temperature, top_p, max_new_tokens)

recovered_image = state.recovered_image
fig = plt.figure()
plt.imshow(recovered_image)
role, message = state.messages[-1]
print(f'{role}: {message}')

ValueError: Image features and image tokens do not match: tokens: 0, features 391

In [8]:
prompt = state.prompt
prompt_len = state.prompt_len
image = state.image

inputs = processor(image, prompt, return_tensors="pt").to(model.device)
input_ids = inputs.input_ids

In [9]:
prompt

'<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\n<image>\nWhat animal is this and what is on the left of it?<|im_end|>\n<|im_start|>assistant\n'

In [10]:
prompt_len

166

In [11]:
inputs

{'input_ids': tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198,     27,   1805,    397,   3838,
           9864,    374,    419,    323,   1128,    374,    389,    279,   2115,
            315,    432,     30, 151645,    198, 151644,  77091,    198]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0'), 'pixel_values': tensor([[ 0.2515,  0.3099,  0.3391,  ..., -0.6270, -0.3995, -0.4990],
        [ 0.5143,  0.2807,  0.5581,  ..., -0.2857, -0.4137, -0.2573],
        [-0.0113,  0.0909, -0.0842,  ..., -1.0963, -1.0252, -0.9683],
        ...,
        [ 1.7114,  1.6238,  1.6238,  ...,  1.1505,  1.0652,  1.0225],
        [ 1.4486,  1.5800,  1.5216,  ..., -0.3426, -0.2146,  0.2688],
        [ 1.6530,  1.6676,  1.5508,  ..., -1.0678, -0.8545, -0.8830]],
       device='cuda:0'), 'image_

In [16]:
len(input_ids[0])

35

In [17]:
torch.where(input_ids==model.config.image_token_index)

(tensor([], device='cuda:0', dtype=torch.int64),
 tensor([], device='cuda:0', dtype=torch.int64))

In [18]:
generated_ids = model.generate(**inputs, max_new_tokens=128)

ValueError: Image features and image tokens do not match: tokens: 0, features 391

In [19]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg",
            },
            {"type": "text", "text": "Describe this image."},
        ],
    }
]

In [20]:
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

In [21]:
text

'<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\n<|vision_start|><|image_pad|><|vision_end|>Describe this image.<|im_end|>\n<|im_start|>assistant\n'

In [23]:
from qwen_vl_utils import process_vision_info

image_inputs, video_inputs = process_vision_info(messages)

In [28]:
type(image_inputs[0])

PIL.Image.Image

In [30]:
type(raw_image)

PIL.JpegImagePlugin.JpegImageFile

In [31]:
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda")

In [33]:
inputs

{'input_ids': tensor([[151644,   8948,    198,  ..., 151644,  77091,    198]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1]], device='cuda:0'), 'pixel_values': tensor([[ 0.8501,  0.8501,  0.8647,  ...,  1.3922,  1.3922,  1.3922],
        [ 0.9376,  0.9376,  0.9376,  ...,  1.4491,  1.4491,  1.4491],
        [ 0.9084,  0.9376,  0.9376,  ...,  1.4065,  1.4207,  1.4207],
        ...,
        [-0.1280, -0.1280, -0.1426,  ..., -0.2431, -0.2715, -0.3000],
        [-0.3324, -0.3324, -0.3032,  ..., -0.3000, -0.2715, -0.2857],
        [-0.3762, -0.4054, -0.4054,  ..., -0.4279, -0.4422, -0.4564]],
       device='cuda:0'), 'image_grid_thw': tensor([[  1,  98, 146]], device='cuda:0')}

In [37]:
inputs.keys()

dict_keys(['input_ids', 'attention_mask', 'pixel_values', 'image_grid_thw'])

In [32]:
generated_ids = model.generate(**inputs, max_new_tokens=128)